# PCT Training

Trains Menghao PCT model from the Point-Transformers implementation: https://github.com/qq456cvb/Point-Transformers

Note: specifically for training it on full PAPNet data.

## Env prep

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# Paths, folders
import os

REPO_PATH = '/content/pointcloud-bench'
DRIVE_PATH = '/content/drive/MyDrive/_Temp/cs7643-final-proj'
results_dir = os.path.join(DRIVE_PATH, 'results')
os.makedirs(results_dir, exist_ok=True)

In [3]:
# Get repo
!git clone --recursive --branch lean-model https://github.com/DavidClaszen/pointcloud-bench {REPO_PATH}
%cd {REPO_PATH}
%pip install -r envs/pct/requirements.txt

Cloning into '/content/pointcloud-bench'...
remote: Enumerating objects: 396, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 396 (delta 29), reused 39 (delta 10), pack-reused 317 (from 1)
Receiving objects: 100% (396/396), 39.81 MiB | 36.33 MiB/s, done.
Resolving deltas: 100% (168/168), done.
Submodule 'repos/PAPNet' (https://github.com/DavidClaszen/PAPNet.git) registered for path 'repos/PAPNet'
Submodule 'repos/Point-Transformers' (https://github.com/DavidClaszen/Point-Transformers.git) registered for path 'repos/Point-Transformers'
Cloning into '/content/pointcloud-bench/repos/PAPNet'...
remote: Enumerating objects: 133, done.        
remote: Counting objects: 100% (133/133), done.        
remote: Compressing objects: 100% (105/105), done.        
remote: Total 133 (delta 55), reused 85 (delta 27), pack-reused 0 (from 0)        
Receiving objects: 100% (133/133), 9.66 MiB | 13.38 MiB/s, done.
Resolving deltas: 100% (

In [49]:
%cd {REPO_PATH}
!git pull
!git submodule update --init

/content/pointcloud-bench
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 1), reused 3 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 321 bytes | 321.00 KiB/s, done.
From https://github.com/DavidClaszen/pointcloud-bench
   aa7ef4b..482a6e8  lean-model -> origin/lean-model
Fetching submodule repos/Point-Transformers
fatal: remote error: upload-pack: not our ref 0a8fa2f5c5b0fef8496b7dbd1be933a4615a5e8f
Errors during submodule fetch:
	repos/Point-Transformers


In [4]:
# Check for CUDA/GPU
import torch, sys
print(sys.version)
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.is_available())

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.9.0+cu126 CUDA: 12.6 GPU: True


In [5]:
# Copy and unzip only the partialmodelnet40 set
# Evaluation will be done in other notebook
!rsync -avP {DRIVE_PATH}/partialmodelnet40.tar.gz {REPO_PATH}/datasets
%cd {REPO_PATH}
!tar -xvzf datasets/partialmodelnet40.tar.gz -C datasets

sending incremental file list
partialmodelnet40.tar.gz
  2,162,130,396 100%  107.68MB/s    0:00:19 (xfr#1, to-chk=0/1)

sent 2,162,658,366 bytes  received 35 bytes  105,495,531.76 bytes/sec
total size is 2,162,130,396  speedup is 1.00
/content/pointcloud-bench
partialmodelnet40/
partialmodelnet40/test_labels.npy
partialmodelnet40/test_gt_tra.npy
partialmodelnet40/test_gt_rot.npy
partialmodelnet40/partialmodelnet40_shape_names.txt
partialmodelnet40/partialmodelnet40_test.txt
partialmodelnet40/partialmodelnet40_train.txt
partialmodelnet40/train_points.npy
partialmodelnet40/train_labels.npy
partialmodelnet40/train_gt_tra.npy
partialmodelnet40/train_gt_rot.npy
partialmodelnet40/test_points.npy


# Model Training

Since we're only using PAPNet style data here, always set `use_papnet_loader` to `True`.


In [22]:
%cd /content/pointcloud-bench/repos/Point-Transformers
!python train_cls.py --help

/content/pointcloud-bench/repos/Point-Transformers
/content/pointcloud-bench/repos/Point-Transformers/train_cls.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)
train_cls is powered by Hydra.

== Configuration groups ==
Compose your configuratio

In [ ]:
# Train Lean
%cd {REPO_PATH}/repos/Point-Transformers
!python train_kd.py model=Lean use_papnet_loader=True batch_size=512 learning_rate=0.0005 epoch=20 workers=4 step_size=5 data_path=../../datasets/partialmodelnet40/ checkpoint_path={DRIVE_PATH}/pct-checkpoint/pct-lean.pth kd.teacher.checkpoint_path={DRIVE_PATH}/pct_final_models/pct_p50_pfull.pth

/content/pointcloud-bench/repos/Point-Transformers
/content/pointcloud-bench/repos/Point-Transformers/train_kd.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1617: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  _C._set_float32_matmul_precision(precision)
/content/pointcloud-bench/repos/Point-Transformers/train_kd.py:22: DeprecationWarning

In [ ]:
+L# Zip Point-Transformers logs, included last best model
!zip -r results.zip ./log/kd/Lean/

  adding: log/cls/Menghao/ (stored 0%)
  adding: log/cls/Menghao/model.py (deflated 76%)
  adding: log/cls/Menghao/train_cls.log (deflated 87%)
  adding: log/cls/Menghao/best_model.pth (deflated 9%)
  adding: log/cls/Menghao/.hydra/ (stored 0%)
  adding: log/cls/Menghao/.hydra/config.yaml (deflated 29%)
  adding: log/cls/Menghao/.hydra/overrides.yaml (deflated 19%)
  adding: log/cls/Menghao/.hydra/hydra.yaml (deflated 66%)
